# 02 — Métricas e hipóteses para o Airbnb

Esta etapa transforma preços anunciados auditados em métricas comparáveis por anúncio e cenários explícitos de potencial bruto anualizado. Não há receita realizada, ocupação observada, recomendação de investimento ou cálculo final de retorno nesta etapa.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 140)


def encontrar_raiz(inicio=None):
    caminho = Path(inicio or Path.cwd()).resolve()
    for candidato in (caminho, *caminho.parents):
        if (candidato / 'data').is_dir() and (candidato / 'ROADMAP.md').is_file():
            return candidato
    raise FileNotFoundError('Raiz do projeto não encontrada.')


def sha256(caminho, tamanho_bloco=1024 * 1024):
    resumo = hashlib.sha256()
    with caminho.open('rb') as arquivo:
        for bloco in iter(lambda: arquivo.read(tamanho_bloco), b''):
            resumo.update(bloco)
    return resumo.hexdigest().upper()


def converter_booleano(serie):
    mapa = {'true': True, 'false': False, '1': True, '0': False}
    return serie.astype('string').str.strip().str.lower().map(mapa).astype('boolean')


RAIZ = encontrar_raiz()
DADOS = RAIZ / 'data'
ENTRADAS = RAIZ / 'outputs' / 'auditoria'
SAIDAS = RAIZ / 'outputs' / 'metricas'
SAIDAS.mkdir(parents=True, exist_ok=True)

HASHES_DADOS = {
    'Details_Itapema.csv': '7A28A35811B5B01CA046D06E0AF80180E43D07AF6923FC03B76DF99AC01050C9',
    'Hosts_ids_Itapema.csv': 'B2E5AA3E0BD30A3FA63643ABC4BC3142C78BE165855BBD6C4D077D6BDE308EA9',
    'Mesh_Ids_Data_Itapema.csv': '7C9DAA0D37FE5C8FA10E6EFA53CB9E6F66E28880E165A62D3E1F9C74585ADF1E',
    'Price_AV_Itapema.csv': 'B0B5C8C07011DAF5C91F2FB9E7BA735026F0AE4542745481376140A714DD813B',
    'VivaReal_Itapema.csv': 'C720320AE6BCD34982323A2D6EEC6D5F5F18E316B3A3DAE0A37F03638E32A631',
}

hashes_antes = {arquivo.name: sha256(arquivo) for arquivo in DADOS.glob('*.csv')}
assert hashes_antes == HASHES_DADOS, 'Os dados brutos diferem da versão validada.'

arquivos_entrada = {
    'airbnb': ENTRADAS / 'airbnb_listings.csv',
    'precos': ENTRADAS / 'precos_airbnb.csv',
}
assert all(caminho.is_file() for caminho in arquivos_entrada.values()), 'Artefatos da Etapa 1 ausentes.'
print(f'Entradas: {ENTRADAS}')
print(f'Saídas permitidas: {SAIDAS}')
print('Dados brutos e artefatos necessários confirmados.')

Entradas: C:\Users\rewel\Documents\jt2026-Antonio-Rewelli-Santos\outputs\auditoria
Saídas permitidas: C:\Users\rewel\Documents\jt2026-Antonio-Rewelli-Santos\outputs\metricas
Dados brutos e artefatos necessários confirmados.


In [2]:
airbnb = pd.read_csv(
    arquivos_entrada['airbnb'],
    dtype={'airbnb_listing_id': 'string', 'owner_id': 'string'},
    low_memory=False,
)
precos = pd.read_csv(
    arquivos_entrada['precos'],
    dtype={'airbnb_listing_id': 'string'},
    low_memory=False,
)

for coluna in ['date', 'aquisition_date', 'primeira_captura', 'ultima_captura']:
    precos[coluna] = pd.to_datetime(precos[coluna], errors='coerce', format='mixed')
for coluna in ['flag_preco_invalido', 'flag_captura_apos_estadia']:
    precos[coluna] = converter_booleano(precos[coluna]).fillna(False)

assert airbnb['airbnb_listing_id'].notna().all()
assert airbnb['airbnb_listing_id'].is_unique
assert not precos.duplicated(['airbnb_listing_id', 'date']).any()
assert {'suburb_key', 'number_of_bedrooms', 'listing_type', 'latitude_analitica', 'longitude_analitica'} <= set(airbnb.columns)

resumo_entradas = pd.DataFrame(
    [
        {'base': 'airbnb_listings', 'linhas': len(airbnb), 'colunas': airbnb.shape[1]},
        {'base': 'precos_airbnb', 'linhas': len(precos), 'colunas': precos.shape[1]},
    ]
)
display(resumo_entradas)

,base,linhas,colunas
0,airbnb_listings,4441,61
1,precos_airbnb,59040,9


In [3]:
ids_airbnb = set(airbnb['airbnb_listing_id'])
trabalho = precos.copy()

mascara_chave_invalida = trabalho['airbnb_listing_id'].isna() | trabalho['date'].isna()
mascara_preco_invalido = trabalho['flag_preco_invalido'] | trabalho['price'].isna() | (trabalho['price'] <= 0)
mascara_captura_apos_estadia = trabalho['flag_captura_apos_estadia'] | (trabalho['aquisition_date'] > trabalho['date'])
mascara_id_orfao = ~trabalho['airbnb_listing_id'].isin(ids_airbnb)

filtros = [
    ('entrada_auditada', pd.Series(True, index=trabalho.index)),
    ('chave_e_data_validas', ~mascara_chave_invalida),
    ('preco_anunciado_positivo', ~mascara_preco_invalido),
    ('captura_nao_posterior_a_estadia', ~mascara_captura_apos_estadia),
    ('id_existente_em_airbnb', ~mascara_id_orfao),
]

mascara_acumulada = pd.Series(True, index=trabalho.index)
funil = []
for etapa, mascara in filtros:
    antes = int(mascara_acumulada.sum())
    mascara_acumulada &= mascara.fillna(False)
    depois = int(mascara_acumulada.sum())
    funil.append({'etapa': etapa, 'linhas_antes': antes, 'linhas_removidas': antes - depois, 'linhas_depois': depois})

funil_filtros = pd.DataFrame(funil)
precos_validos = trabalho.loc[mascara_acumulada].copy()
precos_validos['dia_semana'] = precos_validos['date'].dt.day_name()
precos_validos['fim_de_semana'] = precos_validos['date'].dt.dayofweek >= 5
precos_validos['mes_estadia'] = precos_validos['date'].dt.to_period('M').astype('string')
precos_validos['dias_antecedencia_captura'] = (
    precos_validos['date'] - precos_validos['aquisition_date']
).dt.total_seconds() / 86400

assert not precos_validos.duplicated(['airbnb_listing_id', 'date']).any()
assert precos_validos['price'].gt(0).all()
assert precos_validos['airbnb_listing_id'].isin(ids_airbnb).all()
assert (precos_validos['aquisition_date'] <= precos_validos['date']).all()
display(funil_filtros)

,etapa,linhas_antes,linhas_removidas,linhas_depois
0,entrada_auditada,59040,0,59040
1,chave_e_data_validas,59040,0,59040
2,preco_anunciado_positivo,59040,0,59040
3,captura_nao_posterior_a_estadia,59040,69,58971
4,id_existente_em_airbnb,58971,440,58531


In [4]:
agrupado = precos_validos.groupby('airbnb_listing_id', sort=False)
metricas_preco = agrupado.agg(
    observacoes_preco=('price', 'size'),
    datas_validas=('date', 'nunique'),
    primeira_data_estadia=('date', 'min'),
    ultima_data_estadia=('date', 'max'),
    diaria_mediana_anunciada=('price', 'median'),
    diaria_media_anunciada=('price', 'mean'),
    diaria_minima_anunciada=('price', 'min'),
    diaria_maxima_anunciada=('price', 'max'),
    diaria_desvio_padrao=('price', 'std'),
    capturas_medias_por_data=('quantidade_capturas', 'mean'),
    capturas_maximas_por_data=('quantidade_capturas', 'max'),
    antecedencia_mediana_dias=('dias_antecedencia_captura', 'median'),
).reset_index()

quantis = agrupado['price'].quantile([0.25, 0.75]).unstack().rename(
    columns={0.25: 'diaria_p25_anunciada', 0.75: 'diaria_p75_anunciada'}
).reset_index()
fim_semana = (
    precos_validos.groupby(['airbnb_listing_id', 'fim_de_semana'])['price']
    .median()
    .unstack()
    .rename(columns={False: 'diaria_mediana_dia_util', True: 'diaria_mediana_fim_semana'})
    .reset_index()
)
percentual_fim_semana = (
    agrupado['fim_de_semana'].mean().mul(100).rename('observacoes_fim_semana_percentual').reset_index()
)
metricas_preco = (
    metricas_preco.merge(quantis, on='airbnb_listing_id', validate='one_to_one')
    .merge(fim_semana, on='airbnb_listing_id', how='left', validate='one_to_one')
    .merge(percentual_fim_semana, on='airbnb_listing_id', validate='one_to_one')
)
metricas_preco['diaria_iqr'] = metricas_preco['diaria_p75_anunciada'] - metricas_preco['diaria_p25_anunciada']
metricas_preco['dias_intervalo_observado'] = (
    metricas_preco['ultima_data_estadia'] - metricas_preco['primeira_data_estadia']
).dt.days + 1
metricas_preco['cobertura_intervalo_percentual'] = (
    metricas_preco['datas_validas'] / metricas_preco['dias_intervalo_observado'] * 100
)

colunas_perfil = [
    'airbnb_listing_id', 'ad_name', 'listing_type', 'number_of_bedrooms', 'number_of_bathrooms',
    'number_of_guests', 'suburb', 'suburb_key', 'latitude_analitica', 'longitude_analitica',
    'star_rating', 'number_of_reviews', 'cleaning_fee', 'is_superhost', 'is_professional',
]
metricas_anuncio = airbnb[colunas_perfil].merge(
    metricas_preco, on='airbnb_listing_id', how='left', validate='one_to_one'
)
metricas_anuncio['tem_preco_valido'] = metricas_anuncio['datas_validas'].notna()
metricas_anuncio['observacoes_preco'] = metricas_anuncio['observacoes_preco'].fillna(0).astype('int64')
metricas_anuncio['datas_validas'] = metricas_anuncio['datas_validas'].fillna(0).astype('int64')
for limite in [7, 14, 30]:
    metricas_anuncio[f'amostra_elegivel_{limite}_datas'] = metricas_anuncio['datas_validas'] >= limite
metricas_anuncio['amostra_preco_elegivel'] = metricas_anuncio['amostra_elegivel_14_datas']

assert len(metricas_anuncio) == len(airbnb)
assert metricas_anuncio['airbnb_listing_id'].is_unique
display(metricas_anuncio[['tem_preco_valido', 'amostra_preco_elegivel']].value_counts(dropna=False).rename('anuncios').to_frame())

anuncios
tem_preco_valido amostra_preco_elegivel          
False            False                       3442
True             True                         959
                 False                         40

In [5]:
def cobertura_por_dimensao(df, dimensao, rotulo):
    trabalho_dimensao = df.copy()
    trabalho_dimensao['categoria'] = trabalho_dimensao[dimensao].astype('string').fillna('__ausente__')
    trabalho_dimensao['diaria_elegivel'] = trabalho_dimensao['diaria_mediana_anunciada'].where(
        trabalho_dimensao['amostra_preco_elegivel']
    )
    resultado = trabalho_dimensao.groupby('categoria', dropna=False).agg(
        total_anuncios=('airbnb_listing_id', 'size'),
        anuncios_com_preco=('tem_preco_valido', 'sum'),
        anuncios_elegiveis=('amostra_preco_elegivel', 'sum'),
        diaria_mediana_segmento=('diaria_elegivel', 'median'),
        diaria_p25_entre_anuncios=('diaria_elegivel', lambda serie: serie.quantile(0.25)),
        diaria_p75_entre_anuncios=('diaria_elegivel', lambda serie: serie.quantile(0.75)),
    ).reset_index()
    resultado.insert(0, 'dimensao', rotulo)
    resultado['cobertura_preco_percentual'] = resultado['anuncios_com_preco'] / resultado['total_anuncios'] * 100
    resultado['cobertura_elegivel_percentual'] = resultado['anuncios_elegiveis'] / resultado['total_anuncios'] * 100
    return resultado


global_df = metricas_anuncio.assign(categoria='todos', diaria_elegivel=lambda df: df['diaria_mediana_anunciada'].where(df['amostra_preco_elegivel']))
cobertura_global = global_df.groupby('categoria').agg(
    total_anuncios=('airbnb_listing_id', 'size'),
    anuncios_com_preco=('tem_preco_valido', 'sum'),
    anuncios_elegiveis=('amostra_preco_elegivel', 'sum'),
    diaria_mediana_segmento=('diaria_elegivel', 'median'),
    diaria_p25_entre_anuncios=('diaria_elegivel', lambda serie: serie.quantile(0.25)),
    diaria_p75_entre_anuncios=('diaria_elegivel', lambda serie: serie.quantile(0.75)),
).reset_index()
cobertura_global.insert(0, 'dimensao', 'global')
cobertura_global['cobertura_preco_percentual'] = cobertura_global['anuncios_com_preco'] / cobertura_global['total_anuncios'] * 100
cobertura_global['cobertura_elegivel_percentual'] = cobertura_global['anuncios_elegiveis'] / cobertura_global['total_anuncios'] * 100

cobertura_amostra = pd.concat(
    [
        cobertura_global,
        cobertura_por_dimensao(metricas_anuncio, 'suburb_key', 'bairro'),
        cobertura_por_dimensao(metricas_anuncio, 'number_of_bedrooms', 'numero_quartos'),
        cobertura_por_dimensao(metricas_anuncio, 'listing_type', 'tipo_anuncio'),
    ],
    ignore_index=True,
)
display(cobertura_amostra.head(20))

,dimensao,categoria,total_anuncios,anuncios_com_preco,anuncios_elegiveis,diaria_mediana_segmento,diaria_p25_entre_anuncios,diaria_p75_entre_anuncios,cobertura_preco_percentual,cobertura_elegivel_percentual
0,global,todos,4441,999,959,550.00,400.00,750.0000,22.494934,21.594236
1,bairro,__ausente__,5,3,3,709.00,649.50,754.5000,60.000000,60.000000
2,bairro,altosaobento,62,5,5,280.00,199.00,400.0000,8.064516,8.064516
3,bairro,areal,5,1,1,556.00,556.00,556.0000,20.000000,20.000000
4,bairro,cantodapraia,28,9,8,599.75,508.25,794.7500,32.142857,28.571429
5,bairro,casabranca,88,15,14,349.50,301.50,392.5000,17.045455,15.909091
6,bairro,centro,657,205,200,508.75,383.75,713.5000,31.202435,30.441400
7,bairro,ilhota,56,10,10,495.00,350.00,588.7500,17.857143,17.857143
8,bairro,jardimpraiamar,5,1,1,200.00,200.00,200.0000,20.000000,20.000000
9,bairro,lameiro,1,0,0,NaN,NaN,NaN,0.000000,0.000000


In [6]:
parametros_cenarios = pd.DataFrame(
    [
        {'cenario': 'conservador', 'ocupacao_assumida': 0.40, 'dias_ano': 365, 'natureza': 'hipotese_de_sensibilidade'},
        {'cenario': 'base', 'ocupacao_assumida': 0.55, 'dias_ano': 365, 'natureza': 'hipotese_de_sensibilidade'},
        {'cenario': 'otimista', 'ocupacao_assumida': 0.70, 'dias_ano': 365, 'natureza': 'hipotese_de_sensibilidade'},
    ]
)

elegiveis = metricas_anuncio.loc[
    metricas_anuncio['amostra_preco_elegivel'],
    ['airbnb_listing_id', 'diaria_mediana_anunciada', 'datas_validas'],
].copy()
cenarios_potencial = elegiveis.merge(parametros_cenarios, how='cross')
cenarios_potencial['noites_ocupadas_assumidas'] = (
    cenarios_potencial['dias_ano'] * cenarios_potencial['ocupacao_assumida']
)
cenarios_potencial['potencial_bruto_anualizado'] = (
    cenarios_potencial['diaria_mediana_anunciada'] * cenarios_potencial['noites_ocupadas_assumidas']
).round(2)

def caminho_saida(nome):
    destino = (SAIDAS / nome).resolve()
    assert destino.parent == SAIDAS.resolve(), f'Saída fora do diretório permitido: {destino}'
    return destino


precos_validos.to_csv(caminho_saida('precos_validos.csv'), index=False, encoding='utf-8')
metricas_anuncio.to_csv(caminho_saida('metricas_por_anuncio.csv'), index=False, encoding='utf-8')
cobertura_amostra.to_csv(caminho_saida('cobertura_amostra.csv'), index=False, encoding='utf-8')
parametros_cenarios.to_csv(caminho_saida('parametros_cenarios.csv'), index=False, encoding='utf-8')
cenarios_potencial.to_csv(caminho_saida('cenarios_potencial_bruto.csv'), index=False, encoding='utf-8')

periodo_estadia_inicio = precos_validos['date'].min().date()
periodo_estadia_fim = precos_validos['date'].max().date()
periodo_captura_inicio = precos_validos['aquisition_date'].min()
periodo_captura_fim = precos_validos['aquisition_date'].max()
quantidade_com_preco = int(metricas_anuncio['tem_preco_valido'].sum())
quantidade_elegivel_7 = int(metricas_anuncio['amostra_elegivel_7_datas'].sum())
quantidade_elegivel = int(metricas_anuncio['amostra_preco_elegivel'].sum())
quantidade_elegivel_30 = int(metricas_anuncio['amostra_elegivel_30_datas'].sum())
funil_texto = '\n'.join(
    f"- `{linha.etapa}`: {linha.linhas_depois:,} linha(s) após a regra; {linha.linhas_removidas:,} removida(s) nesta etapa."
    for linha in funil_filtros.itertuples(index=False)
)

metodologia = f'''# Metodologia de métricas e cenários

## Vocabulário

Os valores de `Price_AV` são **preços anunciados de diária**. Não representam reservas, ocupação ou receita realizada. Resultados anualizados são chamados de **potencial bruto anualizado**.

## Universo válido

- Entrada auditada: {len(precos):,} combinações de anúncio/data.
- Saída válida: {len(precos_validos):,} combinações de anúncio/data.
- Anúncios com algum preço válido: {quantidade_com_preco:,} de {len(metricas_anuncio):,}.
- Elegíveis com pelo menos 7 datas: {quantidade_elegivel_7:,}; com 14 datas: {quantidade_elegivel:,}; com 30 datas: {quantidade_elegivel_30:,}.
- Período das estadias: {periodo_estadia_inicio} a {periodo_estadia_fim}.
- Período das capturas utilizadas: {periodo_captura_inicio} a {periodo_captura_fim}.

### Funil de filtros

{funil_texto}
- Outliers não foram removidos; a mediana reduz sua influência.

## Métrica comparável

A métrica primária por anúncio é `diaria_mediana_anunciada`. Segmentos serão comparados pela mediana dessas medianas, para que anúncios com mais datas não recebam peso maior. Sempre devem acompanhar a métrica: número de anúncios, percentis 25/75 e cobertura da amostra.

O corte principal exige 14 datas válidas por anúncio. As flags de 7 e 30 datas permitem testar sensibilidade sem excluir registros da base-mãe.

## Cenários de ocupação

As ocupações de 40%, 55% e 70% são hipóteses de sensibilidade, não estimativas observadas em Itapema.

```text
noites_ocupadas_assumidas = 365 × ocupacao_assumida
potencial_bruto_anualizado = diaria_mediana_anunciada × noites_ocupadas_assumidas
```

A base cobre estadias somente entre {periodo_estadia_inicio} e {periodo_estadia_fim}; multiplicar sua mediana por um ano é uma extrapolação e não captura toda a sazonalidade.

## Fórmulas para a etapa de retorno

```text
investimento_total = preco_compra + custos_aquisicao + mobilia + reforma
custos_operacionais_anuais = custos_variaveis + condominio_anual + IPTU + manutencao + demais_custos_fixos
potencial_liquido_anual = potencial_bruto_anualizado - custos_operacionais_anuais
retorno_liquido = potencial_liquido_anual / investimento_total
payback = investimento_total / potencial_liquido_anual
```

Plataforma, administração, manutenção, condomínio, IPTU, mobília, reforma e custos de aquisição terão entradas separadas. Nenhum valor foi preenchido nesta etapa sem fonte ou hipótese aprovada.

## Limitações

- A cobertura de preços é parcial e pode não ser aleatória.
- Preço anunciado pode diferir do preço efetivamente pago.
- Não há ocupação observada, reservas ou receita realizada.
- Cenários não substituem validação de mercado nem análise completa de sazonalidade.
'''
caminho_saida('metodologia_metricas.md').write_text(metodologia, encoding='utf-8')

display(parametros_cenarios)
display(pd.DataFrame([{'anuncios_elegiveis': quantidade_elegivel, 'linhas_cenarios': len(cenarios_potencial)}]))

,cenario,ocupacao_assumida,dias_ano,natureza
0,conservador,0.40,365,hipotese_de_sensibilidade
1,base,0.55,365,hipotese_de_sensibilidade
2,otimista,0.70,365,hipotese_de_sensibilidade


,anuncios_elegiveis,linhas_cenarios
0,959,2877


In [7]:
hashes_depois = {arquivo.name: sha256(arquivo) for arquivo in DADOS.glob('*.csv')}
assert hashes_depois == hashes_antes == HASHES_DADOS, 'A pasta data/ foi alterada.'
assert len(metricas_anuncio) == len(airbnb)
assert metricas_anuncio['airbnb_listing_id'].is_unique
assert not precos_validos.duplicated(['airbnb_listing_id', 'date']).any()
assert len(cenarios_potencial) == int(metricas_anuncio['amostra_preco_elegivel'].sum()) * len(parametros_cenarios)
recalculado = (
    cenarios_potencial['diaria_mediana_anunciada']
    * cenarios_potencial['dias_ano']
    * cenarios_potencial['ocupacao_assumida']
).round(2)
assert np.allclose(cenarios_potencial['potencial_bruto_anualizado'], recalculado)
assert set(arquivo.name for arquivo in SAIDAS.iterdir()) == {
    'precos_validos.csv',
    'metricas_por_anuncio.csv',
    'cobertura_amostra.csv',
    'parametros_cenarios.csv',
    'cenarios_potencial_bruto.csv',
    'metodologia_metricas.md',
}

display(Markdown(
    f'''## Etapa 2 validada

- **Anúncios totais preservados:** {len(metricas_anuncio):,}.
- **Anúncios com preços válidos:** {int(metricas_anuncio['tem_preco_valido'].sum()):,}.
- **Anúncios elegíveis (14+ datas):** {int(metricas_anuncio['amostra_preco_elegivel'].sum()):,}.
- **Observações válidas de preço:** {len(precos_validos):,}.
- **Cenários por anúncio elegível:** {len(parametros_cenarios)}.
- **Dados brutos:** hashes preservados.
- **Saídas:** restritas a `outputs/metricas/`.
'''
))

## Etapa 2 validada

- **Anúncios totais preservados:** 4,441.
- **Anúncios com preços válidos:** 999.
- **Anúncios elegíveis (14+ datas):** 959.
- **Observações válidas de preço:** 58,531.
- **Cenários por anúncio elegível:** 3.
- **Dados brutos:** hashes preservados.
- **Saídas:** restritas a `outputs/metricas/`.
